# 3. MapReduce

For this exercise we are going to use MapReduce in local mode, i.e. we won't be running anything on the cluster!
 
## 3.1. Use the commands `head`, `cat`, `uniq`, `wc`, `sort`, `find`, `xargs`, `awk` to evaluate the NASA log file:

* Data File:  <https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/nasa/NASA_access_log_Jul95.gz>
* Which page was called the most?
* What was the most frequent return code?
* How many errors occurred? What is the percentage of errors?


In [ ]:
import re
import gzip
import shutil
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm


DATA_DIR = Path("../data")
FILE_NAME = Path("NASA_access_log_Jul95")
LOG_PATTERN = re.compile(r'(\S+) (\S+) (\S+) \[(.*?)\] "(.*?)" (\d{3}) (\d+|-)')
KEYS = "ip, ident, user, date, request, status, bytes".split(", ")
REQUEST_PATTERN = re.compile(r'(\S+)\s+(\S+)(?:\s+(\S+))?')
DOWNLOAD_URL = "https://raw.githubusercontent.com/scalable-infrastructure/exercise-2026/main/data/nasa/NASA_access_log_Jul95.gz"

In [ ]:
def parse_log_line(line: str, pattern=LOG_PATTERN, keys=KEYS) -> dict:
    match = pattern.fullmatch(line.strip())

    if match is None:
        print(f"Invalid log line: {line!r}")
        return {k:None for k in keys}

    return {k:v for k, v in zip(keys, match.groups())}


def load_data(
        file_name: Path = FILE_NAME, 
        data_dir: Path = DATA_DIR, 
        download_url: str = DOWNLOAD_URL, 
        encoding: str = "latin-1",
        as_dataframe = False,
) -> pd.DataFrame | list[str]:

    if not data_dir.exists():
        raise ValueError("Data directory doesn't exist!")

    file_path = data_dir / file_name
    gz_path = file_path.with_suffix(file_path.suffix + ".gz")

    if not file_path.exists():
        # Download data with progress bar
        response = requests.get(download_url, stream=True, timeout=30)
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))

        with open(gz_path, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            desc="Downloading data"
        ) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

        # Unzip
        with gzip.open(gz_path, "rb") as f_in, open(file_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

        # Delete zipped file
        gz_path.unlink()

    # Load data row by row with progress bar
    lines = []
    with open(file_path, "r", encoding=encoding, errors="ignore") as f:
        for line in tqdm(f, desc="Reading file"):
            lines.append(line)

    if as_dataframe:
        data = list(map(parse_log_line, lines))
        return pd.DataFrame(data)
    
    return lines

In [ ]:
df = load_data(as_dataframe=True)
df.head()

Reading file: 0it [01:13, ?it/s]


KeyboardInterrupt: 

Which page was called the most?

In [ ]:
def parse_request_field(req: str, req_pattern=REQUEST_PATTERN) -> dict:
    req_match = req_pattern.fullmatch(req.strip())
    method, path, protocol = req_match.groups()
    return {
        "method": method,
        "path": path,
        "protocol": protocol,
    }

df["request"] = list(map(parse_request_field, requests))
df

ValueError: Length of values (10) does not match length of index (1891715)